# Phi-4 Multimodal — DIMER multi-capability tutorial

**Profile:** `MULTI-CAPABILITY`  
**Capability:** text-, image-, and audio-conditioned text generation using one pinned Phi-4 Multimodal checkpoint

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API rather than reimplementing model inference. The default sample is demonstration evidence, not a production-quality or benchmark claim.

**Learning objectives:** resolve the immutable upstream model revision, validate a public/default input, run the supported task, inspect task-appropriate outputs, exercise an optional BYOD path, and export machine-readable outputs plus provenance.


## Prerequisites

Run in a fresh supported runtime. Install dependencies before importing PyTorch or Transformers. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.


## 1. Bootstrap the repository and pinned runtime

The installation cell installs this repository and its exact model-facing dependency versions. If installation replaces a pre-imported core framework, restart the runtime before continuing.

In [ ]:
%pip install -q -e ".[tutorial]"
import platform, torch, transformers
print({'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})
assert torch.cuda.is_available(), 'A CUDA GPU runtime is required for the release-reference path.'

## 2. Resolve the pinned model and acknowledge the trust boundary

This upstream model requires custom Python code. The repository loader refuses implicit trust; this cell opts in only after printing the immutable code/weight revision.

In [ ]:
from phi4_multimodal_pipeline import MODEL_ID, MODEL_REVISION, Phi4MultimodalPipeline
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'trust_remote_code': 'explicitly enabled at pinned revision'})
pipe = Phi4MultimodalPipeline.from_pretrained(allow_remote_code=True, device='cuda')

## 3. Capability A — text-only generation

Deterministic decoding is used for the tutorial. The result is generated text, not a calibrated confidence statement.

In [ ]:
text_result = pipe.generate('In two sentences, explain what automatic speech recognition does.', max_new_tokens=96, temperature=0.0)
print(text_result['text'])

## 4. Capability B — image + text

The default image is the public stop-sign photograph used in Microsoft's upstream sample. BYOD can replace it by setting `USE_BYOD_IMAGE=True`.

In [ ]:
from PIL import Image
from io import BytesIO
from urllib.request import urlopen
USE_BYOD_IMAGE = False
if USE_BYOD_IMAGE:
    from google.colab import files
    uploaded = files.upload(); image = Image.open(next(iter(uploaded)))
else:
    image = Image.open(BytesIO(urlopen('https://www.ilankelman.org/stopsigns/australia.jpg', timeout=30).read())).convert('RGB')
image_result = pipe.generate('Describe the most prominent traffic sign in the image.', images=[image], max_new_tokens=96, temperature=0.0)
print(image_result['text'])

## 5. Capability C — audio + text

The default path uses a public LibriSpeech sample. The capability produces generated transcription text; no universal accuracy claim is inferred from this one example.

In [ ]:
from datasets import load_dataset
ds = load_dataset('hf-internal-testing/librispeech_asr_dummy','clean',split='validation')
a = ds[0]['audio']
audio = (a['array'], a['sampling_rate'])
audio_result = pipe.generate('Generate a transcription of the attached speech.', audios=[audio], max_new_tokens=128, temperature=0.0)
print(audio_result['text'])

## 6. Export outputs and provenance

Each capability is exported separately while preserving the shared model identity and decoding settings.

In [ ]:
import json, os
os.makedirs('outputs',exist_ok=True)
payload={'model_id':MODEL_ID,'model_revision':MODEL_REVISION,'capabilities':{'text':text_result,'image':image_result,'audio':audio_result},'runtime':{'python':platform.python_version(),'torch':torch.__version__,'transformers':transformers.__version__}}
with open('outputs/phi4_multimodal_results.json','w',encoding='utf-8') as f: json.dump(payload,f,indent=2,ensure_ascii=False)
print('outputs/phi4_multimodal_results.json')

## Interpretation and limits

The three outputs share one generative model but have different evidence sources and failure modes. Non-empty text only establishes that the inference path executed. Image and audio interpretations can be wrong, and no calibrated answer confidence is returned. The tutorial deliberately does not expose fine-tuning or claim that upstream benchmark results were reproduced.

Successful execution proves that this repository revision can acquire the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

## References

- Upstream model: https://huggingface.co/microsoft/Phi-4-multimodal-instruct
- Upstream sample code: https://huggingface.co/microsoft/Phi-4-multimodal-instruct/blob/93f923e1a7727d1c4f446756212d9d3e8fcc5d81/sample_inference_phi4mm.py
- Technical report: https://arxiv.org/abs/2503.01743
